# Alpha enhancement LGBM report

This notebook is a presentation-oriented and reproducible view of the current raw-alpha LGBM research run. It separates:

1. Data integration and label semantics.
2. LGBM training and feature contract.
3. Evaluation, charts, and rank-bucket NAV.

Default behavior is read-only: the notebook loads existing artifacts under `analysis/outputs`. It does not rebuild data, retrain models, or regenerate large evaluation outputs unless a future cell is explicitly changed to call the existing project scripts.

Main run covered here:

`lgbm_non_neutralized_alpha_raw_return_monotone_signal_depth7_leaves127_min250_reg_v1`


## 00 Parameters and run contract

The notebook uses repository-relative paths and environment variables. No personal absolute paths are required. Missing required artifacts fail loudly rather than silently falling back to another run.


In [ ]:
from __future__ import annotations

from pathlib import Path
import importlib.util
import json
import math
import os
import textwrap
import warnings

import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "analysis").exists():
            return candidate
    return start


REPO_ROOT = Path(os.environ.get("QAE_REPO_ROOT", find_repo_root())).resolve()
OUTPUT_ROOT = Path(os.environ.get("QAE_OUTPUT_ROOT", REPO_ROOT / "analysis" / "outputs")).resolve()
MODEL_RUN_ID = os.environ.get(
    "QAE_MODEL_RUN_ID",
    "lgbm_non_neutralized_alpha_raw_return_monotone_signal_depth7_leaves127_min250_reg_v1",
)
DATA_RUN_ID = os.environ.get("QAE_DATA_RUN_ID", "non_neutralized_alpha_raw_return_v1")
SELECTED_BUCKETS = [1, 16, 31]

REBUILD_DATA = False
RETRAIN_MODEL = False
RERUN_METRICS = False
RERUN_RANK_BUCKET = False
RERUN_BIG_CHARTS = False

PROJECT_DEPENDENCIES = {
    name: importlib.util.find_spec(name) is not None
    for name in ["pandas", "numpy", "matplotlib", "seaborn", "IPython", "nbformat", "nbconvert", "ipykernel"]
}


def load_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"), parse_constant=lambda _constant: None)


def read_csv(path: Path, **kwargs) -> pd.DataFrame:
    return pd.read_csv(path, encoding="utf-8-sig", **kwargs)


def require_path(path: Path, label: str) -> Path:
    if not path.exists():
        raise FileNotFoundError(f"Required artifact is missing for {label}: {path}")
    return path


def show(obj, *, max_rows: int = 40) -> None:
    try:
        from IPython.display import display
    except Exception:
        display = None
    if isinstance(obj, pd.DataFrame):
        out = obj.copy()
        if len(out) > max_rows:
            out = out.head(max_rows)
            print(f"Showing first {max_rows} of {len(obj)} rows")
        if display is None:
            print(out.to_string(index=False))
        else:
            display(out)
    else:
        if display is None:
            print(obj)
        else:
            display(obj)


TOKENS = {
    "surface": "#FCFCFD",
    "panel": "#FFFFFF",
    "ink": "#1F2430",
    "muted": "#6F768A",
    "grid": "#E6E8F0",
    "axis": "#D7DBE7",
}
COLOR_FAMILIES = {
    "blue": {"base": "#A3BEFA", "mid": "#5477C4", "dark": "#2E4780", "light": "#CEDFFE", "xlight": "#EAF1FE"},
    "gold": {"base": "#FFE15B", "mid": "#B8A037", "dark": "#736422", "light": "#FFEA8F", "xlight": "#FFF4C2"},
    "orange": {"base": "#F0986E", "mid": "#CC6F47", "dark": "#804126", "light": "#FFBDA1", "xlight": "#FFEDDE"},
    "olive": {"base": "#A3D576", "mid": "#71B436", "dark": "#386411", "light": "#BEEB96", "xlight": "#D8ECBD"},
    "pink": {"base": "#F390CA", "mid": "#BD569B", "dark": "#8A3A6F", "light": "#F5BACC", "xlight": "#FCDAD6"},
}


def use_chart_theme() -> None:
    plt.rcParams.update(
        {
            "figure.facecolor": TOKENS["surface"],
            "axes.facecolor": TOKENS["panel"],
            "axes.edgecolor": TOKENS["axis"],
            "axes.labelcolor": TOKENS["ink"],
            "xtick.color": TOKENS["muted"],
            "ytick.color": TOKENS["muted"],
            "grid.color": TOKENS["grid"],
            "grid.linewidth": 0.8,
            "font.family": ["Aptos", "Inter", "Segoe UI", "DejaVu Sans", "Arial", "sans-serif"],
            "axes.spines.top": False,
            "axes.spines.right": False,
        }
    )


def add_chart_header(fig, ax, title: str, subtitle: str) -> None:
    if not title or not subtitle:
        raise ValueError("Every notebook chart needs a non-empty title and subtitle.")
    ax.set_title("")
    fig.subplots_adjust(top=0.82)
    left = ax.get_position().x0
    fig.text(left, 0.98, textwrap.fill(title, 86), ha="left", va="top", fontsize=13, fontweight="semibold", color=TOKENS["ink"])
    fig.text(left, 0.925, textwrap.fill(subtitle, 120), ha="left", va="top", fontsize=9, color=TOKENS["muted"])


def format_date_axis(ax, *, max_ticks: int = 7) -> None:
    locator = mdates.AutoDateLocator(minticks=3, maxticks=max_ticks)
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))
    ax.tick_params(axis="x", labelrotation=0)


def pct(value: float | int | None) -> str:
    if value is None or (isinstance(value, float) and not math.isfinite(value)):
        return "n/a"
    return f"{float(value) * 100:.2f}%"


use_chart_theme()
print(f"Repo root: {REPO_ROOT}")
print(f"Output root: {OUTPUT_ROOT}")
print(f"Model run: {MODEL_RUN_ID}")
print("Dependency availability:")
for name, available in PROJECT_DEPENDENCIES.items():
    print(f"  {name}: {available}")
if not PROJECT_DEPENDENCIES["seaborn"]:
    print("Seaborn is not installed in the current .venv; notebook charts use explicit Matplotlib styling only.")


In [ ]:
DATA_DIR = OUTPUT_ROOT / "training_data" / DATA_RUN_ID
MODEL_DIR = OUTPUT_ROOT / MODEL_RUN_ID
DETAIL_DIR = MODEL_DIR / "detailed_metrics"
RB_FOLD_DIR = MODEL_DIR / "rank_bucket_pred_direct_31_v1"
CHRONO_PRED_DIR = MODEL_DIR / "chronological_pred_direct_full_v1"
RB_CHRONO_DIR = MODEL_DIR / "rank_bucket_pred_direct_31_full_chronological_v1"
PERIOD_DIR = OUTPUT_ROOT / "full_chrono_period_segments_v1"
ALPHA_COMPARE_DIR = OUTPUT_ROOT / "alpha_signal_rank_bucket_comparison_strict_no_liq_v1"

PATHS = {
    "data_summary": DATA_DIR / "training_summary.json",
    "data_split_summary": DATA_DIR / "diagnostics" / "split_summary.csv",
    "data_feature_diagnostics": DATA_DIR / "diagnostics" / "feature_standardization_diagnostics.csv",
    "data_return_distribution": DATA_DIR / "diagnostics" / "raw_return_distribution_by_date.csv",
    "model_summary": MODEL_DIR / "training_summary.json",
    "compact_metrics": MODEL_DIR / "metrics_by_fold_split.csv",
    "detailed_evaluation_summary": DETAIL_DIR / "evaluation_summary.json",
    "overall_metrics": DETAIL_DIR / "overall_metrics_by_fold_split_score.csv",
    "daily_ic": DETAIL_DIR / "daily_ic_by_fold_split_score.csv",
    "top_bottom_by_date": DETAIL_DIR / "top_bottom_spread_by_date.csv",
    "top_bottom_summary": DETAIL_DIR / "top_bottom_spread_summary.csv",
    "feature_gain_summary": DETAIL_DIR / "feature_gain_summary.csv",
    "feature_gain_role_summary": DETAIL_DIR / "feature_gain_role_summary.csv",
    "fold_rank_bucket_summary": RB_FOLD_DIR / "rank_bucket_summary.csv",
    "full_chrono_manifest": CHRONO_PRED_DIR / "selection_manifest.json",
    "full_chrono_summary": RB_CHRONO_DIR / "rank_bucket_summary.csv",
    "full_chrono_nav": RB_CHRONO_DIR / "rank_bucket_nav.csv",
    "full_chrono_analysis": RB_CHRONO_DIR / "chronological_nav_analysis_summary.json",
    "full_chrono_rank_bucket_eval": RB_CHRONO_DIR / "rank_bucket_evaluation_summary.json",
    "period_requested_table": PERIOD_DIR / "rank_bucket_period_requested_table.csv",
    "period_summary": PERIOD_DIR / "rank_bucket_period_summary.csv",
    "period_evaluation_summary": PERIOD_DIR / "rank_bucket_period_evaluation_summary.json",
}

for label, path in PATHS.items():
    require_path(path, label)

data_summary = load_json(PATHS["data_summary"])
model_summary = load_json(PATHS["model_summary"])
eval_summary = load_json(PATHS["detailed_evaluation_summary"])
chrono_manifest = load_json(PATHS["full_chrono_manifest"])
chrono_analysis = load_json(PATHS["full_chrono_analysis"])
rank_bucket_eval = load_json(PATHS["full_chrono_rank_bucket_eval"])
period_eval = load_json(PATHS["period_evaluation_summary"])

assert eval_summary["schema_version"] == "lgbm_training_metrics_v1"
assert chrono_manifest["schema_version"] == "chronological_rank_bucket_predictions_v1"
assert chrono_manifest["score_col"] == "pred_direct"
assert chrono_manifest["output_fold_id"] == 0
assert chrono_manifest["output_split"] == "full_chronological"
assert chrono_manifest["selection_policy"] == "first_fold_train_valid_test_then_later_fold_tests"
assert chrono_manifest["row_counts"]["selected_dates"] == 658
assert chrono_manifest["row_counts"]["selected_prediction_rows"] == chrono_manifest["row_counts"]["selected_unique_date_stock_rows"]

rank_contract = rank_bucket_eval["metric_contract"]
assert rank_bucket_eval["schema_version"] == "lgbm_rank_bucket_nav_v1"
assert rank_bucket_eval["rank_bucket_count"] == 31
assert rank_contract["score_col"] == "pred_direct"
assert rank_contract["return_col"] == "return_y_hfq_adj"
assert rank_contract["rank_order"] == "descending"
assert rank_contract["bucket_mode"] == "daily_equal_count"
assert rank_contract["cost_model"] == "gross_no_cost"

expected_periods = ["fold1_train", "fold1_valid", "fold1_test", "fold2_test", "fold3_test"]
assert period_eval["schema_version"] == "lgbm_rank_bucket_period_summary_v1"
assert period_eval["bucket_count"] == 31
assert [period["label"] for period in period_eval["periods"]] == expected_periods

artifact_table = pd.DataFrame(
    {"artifact": label, "path": str(path.relative_to(REPO_ROOT)), "exists": path.exists()} for label, path in PATHS.items()
)
show(artifact_table, max_rows=100)


## 01 Data integration and label semantics

The training panel combines same-signal-date raw alpha information with exposure/context fields. The fit target is the raw forward return column `y_return_hfq_adj_fwd`; it is not residualized, not winsorized, and not rank-transformed by the builder.


In [ ]:
metadata = data_summary["metadata"]
panel = data_summary["panel"]
label_diag = data_summary["label_diagnostics"]
standardization = panel["standardization"]

contract_rows = [
    ("signal_stage", metadata["signal_stage"]),
    ("alpha_source", metadata["alpha_source"]),
    ("raw_feature_source", metadata["raw_feature_source"]),
    ("model_form", metadata["model_form"]),
    ("fit_target", metadata["fit_target_col"]),
    ("label_contract", metadata["label_contract"]),
    ("return_neutralization", panel["return_neutralization"]["enabled"]),
    ("alpha_signal_neutralization", panel["alpha_signal_neutralization"]["enabled"]),
    ("feature_asof", metadata["feature_asof"]),
    ("feature_transform_universe", panel["feature_transform_universe"]),
    ("sample_weight_universe", panel["sample_weight_universe"]),
    ("target_transform", panel["target_transform"]),
    ("rank_convention", standardization["rank_convention"]),
    ("z_convention", standardization["z_convention"]),
]
show(pd.DataFrame(contract_rows, columns=["field", "value"]), max_rows=100)


In [ ]:
panel_counts = pd.DataFrame(
    [
        ("raw_feature_row_count", panel["raw_feature_row_count"]),
        ("dropped_missing_label_rows", panel["dropped_missing_label_rows"]),
        ("dropped_missing_feature_rows", panel["dropped_missing_feature_rows"]),
        ("final_panel_row_count", panel["row_count"]),
        ("date_count", panel["date_count"]),
        ("stock_count", panel["stock_count"]),
        ("date_min", panel["date_min"]),
        ("date_max", panel["date_max"]),
        ("raw_return_matrix_shape", str(label_diag["raw_return_shape"])),
        ("raw_return_date_min", label_diag["raw_return_date_min"]),
        ("raw_return_date_max", label_diag["raw_return_date_max"]),
    ],
    columns=["metric", "value"],
)
expected_panel_rows = panel["raw_feature_row_count"] - panel["dropped_missing_label_rows"] - panel["dropped_missing_feature_rows"]
assert expected_panel_rows == panel["row_count"], "Panel row accounting does not match training summary."
assert metadata["neutralization_policy"]["return_y"] == "not_neutralized"
assert metadata["neutralization_policy"]["alpha_signal"] == "not_neutralized"
assert panel["return_neutralization"]["enabled"] is False
assert panel["alpha_signal_neutralization"]["enabled"] is False
show(panel_counts, max_rows=100)


In [ ]:
role_rows = []
for role, columns in data_summary["feature_roles"].items():
    role_rows.append({"role": role, "column_count": len(columns), "columns": ", ".join(columns)})
role_table = pd.DataFrame(role_rows)
show(role_table, max_rows=100)

excluded = set(data_summary["feature_roles"]["excluded_from_model"])
model_features = set(model_summary["feature_columns"])
assert model_features.isdisjoint(excluded), "A model feature is also listed as excluded_from_model."
assert model_summary["target_col"] not in model_features, "Target leaked into feature columns."
assert "factor_sss_dx_10_raw" in excluded
assert "sample_weight" in excluded


In [ ]:
split_summary = read_csv(PATHS["data_split_summary"])
feature_diag = read_csv(PATHS["data_feature_diagnostics"])
raw_return_dist = read_csv(PATHS["data_return_distribution"])

print("Fold/split row diagnostics from data builder:")
show(split_summary, max_rows=100)
print("Feature standardization diagnostics sample:")
show(feature_diag.head(12), max_rows=12)
print("Raw return distribution sample:")
show(raw_return_dist.head(8), max_rows=8)


Data integration guardrails for interpretation:

- Do not describe this target as `y_resid_fwd` or neutralized return.
- Do not put `factor_sss_dx_10_raw`, `market_cap`, `log_mcap`, target columns, or `sample_weight` into the model feature set.
- Do not recompute alpha rank/z after label join; the documented universe is same-date raw feature rows before label join.
- Do not shift returns in this notebook; the return matrix is assumed to be already aligned to signal dates by the upstream data artifact.


## 02 LGBM training and feature contract

The model trains LightGBM on raw forward returns using two raw-alpha signal transforms plus context/style features. The monotone constraint is local partial monotonicity with other features fixed; it is not a guarantee of global rank-bucket ordering across different contexts.


In [ ]:
constraint_by_feature = model_summary["monotone_constraints"]["constraints_by_feature"]
feature_roles = model_summary["feature_roles"]

feature_rows = []
for feature in model_summary["feature_columns"]:
    if feature in model_summary["signal_features"]:
        role = "signal_features"
    elif feature in model_summary["condition_categorical_features"]:
        role = "condition_categorical"
    elif feature in model_summary["condition_continuous_features"]:
        role = "condition_continuous"
    else:
        role = "unknown"
    feature_rows.append(
        {
            "feature": feature,
            "role": role,
            "categorical": feature in model_summary["categorical_features"],
            "monotone_constraint": constraint_by_feature.get(feature, None),
        }
    )
feature_table = pd.DataFrame(feature_rows)
assert feature_table["feature"].tolist() == model_summary["feature_columns"]
assert model_summary["monotone_constraints"]["constrained_features"] == model_summary["signal_features"]
show(feature_table, max_rows=100)


In [ ]:
fold_rows = []
for fold in model_summary["folds"]:
    row = {
        "fold_id": fold["fold_id"],
        "best_iteration": fold["best_iteration"],
        "valid_huber": fold["best_score"].get("valid", {}).get("huber"),
    }
    for split in ["train", "valid", "test"]:
        date_range = fold["date_ranges"][split]
        row[f"{split}_start"] = date_range["start"]
        row[f"{split}_end"] = date_range["end"]
        row[f"{split}_date_count"] = date_range["date_count"]
        row[f"{split}_rows"] = fold["rows"][split]
    fold_rows.append(row)
fold_table = pd.DataFrame(fold_rows)
show(fold_table, max_rows=20)

if (fold_table["best_iteration"] <= 1).any():
    print("Attention: at least one fold stopped at iteration 1; treat this as a model-diagnostics item, not as a notebook error.")


In [ ]:
important_params = [
    "objective", "learning_rate", "n_estimators", "num_leaves", "max_depth", "min_child_samples",
    "subsample", "colsample_bytree", "reg_alpha", "reg_lambda", "cat_smooth", "cat_l2", "random_state",
]
params_table = pd.DataFrame(
    {"parameter": key, "value": model_summary["model_params"].get(key)} for key in important_params
)
monotone_table = pd.DataFrame(
    {"feature": feature, "constraint": constraint_by_feature[feature]}
    for feature in model_summary["feature_columns"]
)
print(f"Early stopping rounds: {model_summary['early_stopping_rounds']}")
show(params_table, max_rows=100)
show(monotone_table, max_rows=100)


In [ ]:
score_definitions = pd.DataFrame(
    [
        {"score_column": "pred_direct", "definition": "Same trained model prediction using signal and context features: g(signal, context)."},
        {"score_column": "pred_context_only", "definition": "Counterfactual prediction from the same model after setting signal features to 0 while preserving context."},
        {"score_column": "score_marginal", "definition": "pred_direct - pred_context_only; model marginal contribution relative to a zero-signal baseline."},
        {"score_column": "score_marginal_z", "definition": "Per-date robust-z transform of score_marginal. Current IC tables have zero valid date_count for this run."},
    ]
)
show(score_definitions, max_rows=20)


## 03 Evaluation, charts, and rank-bucket NAV

Evaluation is shown in three layers:

1. Cross-sectional IC/RankIC and top-bottom spread for prediction quality.
2. Feature gain diagnostics for model inspection.
3. Rank-bucket NAV for equal-weight gross return paths.

The headline score for this notebook is `pred_direct`. The current run's `score_marginal_z` IC tables have zero valid dates, so it is not used as a headline evaluation signal.


In [ ]:
compact_metrics = read_csv(PATHS["compact_metrics"])
overall_metrics = read_csv(PATHS["overall_metrics"])
daily_ic = read_csv(PATHS["daily_ic"], parse_dates=["date"])
top_bottom_summary = read_csv(PATHS["top_bottom_summary"])
top_bottom_by_date = read_csv(PATHS["top_bottom_by_date"], parse_dates=["date"])
feature_gain = read_csv(PATHS["feature_gain_summary"])
feature_gain_role = read_csv(PATHS["feature_gain_role_summary"])

score_z_rows = compact_metrics[compact_metrics["prediction_col"].eq("score_marginal_z")]
assert score_z_rows["date_count"].fillna(0).astype(int).sum() == 0, "score_marginal_z unexpectedly has valid compact metric dates."
print("Compact metrics, excluding score_marginal_z because its date_count is zero in this run:")
show(compact_metrics[~compact_metrics["prediction_col"].eq("score_marginal_z")], max_rows=100)
print("Feature gain by role:")
show(feature_gain_role, max_rows=20)


In [ ]:
plot_df = overall_metrics[
    overall_metrics["score_col"].isin(["pred_direct", "score_marginal"])
    & overall_metrics["split"].eq("test")
].copy()
plot_df["label"] = "fold " + plot_df["fold_id"].astype(str) + " / " + plot_df["score_col"]
plot_df = plot_df.sort_values(["fold_id", "score_col"])

fig, ax = plt.subplots(figsize=(10, 4.8))
colors = [COLOR_FAMILIES["blue"]["base"] if s == "pred_direct" else COLOR_FAMILIES["gold"]["base"] for s in plot_df["score_col"]]
edges = [COLOR_FAMILIES["blue"]["dark"] if s == "pred_direct" else COLOR_FAMILIES["gold"]["dark"] for s in plot_df["score_col"]]
ax.barh(plot_df["label"], plot_df["mean_rankic"], color=colors, edgecolor=edges, linewidth=1.0)
ax.axvline(0, color=TOKENS["ink"], linewidth=1.0)
ax.set_xlabel("Mean daily RankIC")
ax.grid(axis="x", color=TOKENS["grid"])
add_chart_header(
    fig,
    ax,
    "Fold test RankIC uses pred_direct as the headline score",
    "Mean daily cross-sectional RankIC on y_return_hfq_adj_fwd; score_marginal is shown only as a secondary decomposition.",
)
plt.show()


In [ ]:
daily_plot = daily_ic[(daily_ic["score_col"].eq("pred_direct")) & (daily_ic["split"].eq("test"))].copy()
fig, ax = plt.subplots(figsize=(11, 4.8))
for fold_id, part in daily_plot.groupby("fold_id", sort=True):
    part = part.sort_values("date")
    color = COLOR_FAMILIES[["blue", "gold", "olive"][int(fold_id) - 1]]["base"]
    ax.plot(part["date"], part["rank_ic"].rolling(10, min_periods=1).mean(), label=f"fold {fold_id}", color=color, linewidth=1.2)
ax.axhline(0, color=TOKENS["ink"], linewidth=1.0, linestyle=":")
ax.set_ylabel("10-day rolling RankIC")
ax.grid(True, axis="y")
format_date_axis(ax)
ax.legend(loc="lower left", bbox_to_anchor=(0, 1.02), frameon=False, ncol=3, borderaxespad=0)
add_chart_header(
    fig,
    ax,
    "Rolling test RankIC for pred_direct",
    "Each point is a 10-signal-day rolling mean of daily cross-sectional RankIC within the fold test split.",
)
plt.show()


In [ ]:
spread_plot = top_bottom_by_date[(top_bottom_by_date["score_col"].eq("pred_direct")) & (top_bottom_by_date["split"].eq("test"))].copy()
fig, ax = plt.subplots(figsize=(11, 4.8))
for fold_id, part in spread_plot.groupby("fold_id", sort=True):
    part = part.sort_values("date")
    color = COLOR_FAMILIES[["blue", "gold", "olive"][int(fold_id) - 1]]["base"]
    ax.plot(part["date"], part["spread"].rolling(10, min_periods=1).mean(), label=f"fold {fold_id}", color=color, linewidth=1.2)
ax.axhline(0, color=TOKENS["ink"], linewidth=1.0, linestyle=":")
ax.set_ylabel("10-day rolling top-bottom spread")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.grid(True, axis="y")
format_date_axis(ax)
ax.legend(loc="lower left", bbox_to_anchor=(0, 1.02), frameon=False, ncol=3, borderaxespad=0)
add_chart_header(
    fig,
    ax,
    "Top-bottom return spread is computed from pred_direct quantiles",
    "Spread is mean target return in the top score quantile minus the bottom score quantile; groups are formed without future returns.",
)
plt.show()


In [ ]:
fg = feature_gain.sort_values("mean_gain_share", ascending=True).copy()
fig, ax = plt.subplots(figsize=(10, 6.5))
bar_color = COLOR_FAMILIES["orange"]["base"]
edge_color = COLOR_FAMILIES["orange"]["dark"]
ax.barh(fg["feature"], fg["mean_gain_share"], color=bar_color, edgecolor=edge_color, linewidth=1.0)
ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.set_xlabel("Mean gain share")
ax.grid(axis="x", color=TOKENS["grid"])
add_chart_header(
    fig,
    ax,
    "Feature gain is concentrated in context and alpha signal features",
    "LightGBM gain shares are model split statistics, not causal effects or standalone factor returns.",
)
plt.show()


### Rank-bucket NAV contract

Rank buckets are assigned before looking at realized returns. Current bucket NAV uses `pred_direct`, descending rank order, 31 daily equal-count buckets, and equal-weight valid returns within each bucket. The result is gross no-cost NAV: no fee, no slippage, and no transaction-cost adjustment.


In [ ]:
fold_bucket_summary = read_csv(PATHS["fold_rank_bucket_summary"])
chrono_summary = read_csv(PATHS["full_chrono_summary"])
chrono_nav = read_csv(PATHS["full_chrono_nav"], parse_dates=["signal_date"])
period_requested = read_csv(PATHS["period_requested_table"])
period_requested["bucket_label_2d"] = pd.to_numeric(period_requested["bucket"], errors="raise").astype(int).map(lambda value: f"{value:02d}")

required_nav_columns = {"fold_id", "split", "score_col", "return_col", "signal_date", "bucket_index", "bucket_count", "bucket_mode", "gross_nav"}
required_summary_columns = {"bucket_index", "bucket_label", "date_count", "mean_daily_return", "annualized_return", "sharpe", "max_drawdown", "positive_rate"}
required_period_columns = {"period_label", "period_start", "period_end", "bucket", "bucket_label_2d", "ann_return", "nav_end", "sharpe"}
assert required_nav_columns.issubset(chrono_nav.columns), f"Missing NAV columns: {required_nav_columns - set(chrono_nav.columns)}"
assert required_summary_columns.issubset(chrono_summary.columns), f"Missing summary columns: {required_summary_columns - set(chrono_summary.columns)}"
assert required_period_columns.issubset(period_requested.columns), f"Missing period columns: {required_period_columns - set(period_requested.columns)}"
assert set(chrono_nav["score_col"].unique()) == {"pred_direct"}
assert set(chrono_nav["return_col"].unique()) == {"return_y_hfq_adj"}
assert set(chrono_nav["split"].unique()) == {"full_chronological"}
assert int(chrono_nav["bucket_count"].max()) == 31
assert set(SELECTED_BUCKETS).issubset(set(chrono_summary["bucket_index"]))

selected_summary = chrono_summary[chrono_summary["bucket_index"].isin(SELECTED_BUCKETS)].copy()
selected_summary["final_nav"] = selected_summary["bucket_index"].map(
    chrono_nav.sort_values("signal_date").groupby("bucket_index")["gross_nav"].last()
)
show(
    selected_summary[
        ["bucket_label", "date_count", "mean_daily_return", "annualized_return", "sharpe", "max_drawdown", "positive_rate", "final_nav"]
    ],
    max_rows=20,
)

print("Full chronological selection policy:")
show(pd.DataFrame(chrono_manifest["selected_segments"]), max_rows=20)


In [ ]:
selected_nav = chrono_nav[chrono_nav["bucket_index"].isin(SELECTED_BUCKETS)].copy()
fig, ax = plt.subplots(figsize=(11, 5.4))
colors = {1: COLOR_FAMILIES["blue"]["base"], 16: COLOR_FAMILIES["gold"]["base"], 31: COLOR_FAMILIES["orange"]["base"]}
for bucket_index, part in selected_nav.groupby("bucket_index", sort=True):
    part = part.sort_values("signal_date")
    ax.plot(part["signal_date"], part["gross_nav"], label=f"bucket {bucket_index:02d}", color=colors.get(bucket_index, COLOR_FAMILIES["blue"]["base"]), linewidth=1.4)
for segment in chrono_manifest["selected_segments"]:
    start = pd.to_datetime(segment["start"])
    end = pd.to_datetime(segment["end"])
    if segment["split"] != "test":
        ax.axvspan(start, end, color="#F4F5F7", alpha=0.6, zorder=0)
ax.axhline(1.0, color=TOKENS["ink"], linewidth=1.0, linestyle=":")
ax.set_ylabel("Gross NAV")
ax.grid(True, axis="y")
format_date_axis(ax)
ax.legend(loc="lower left", bbox_to_anchor=(0, 1.02), frameon=False, ncol=3, borderaxespad=0)
add_chart_header(
    fig,
    ax,
    "Full chronological diagnostic NAV separates top, middle, and bottom buckets",
    "Shaded regions are fold1 train/valid segments and are not pure OOS; NAV is equal-weight gross no-cost using pred_direct.",
)
plt.show()


In [ ]:
period_top_bottom = period_requested[period_requested["bucket_label_2d"].isin(["01", "31"])].copy()
period_top_bottom["ann_return_pct"] = period_top_bottom["ann_return"].astype(float) * 100
fig, ax = plt.subplots(figsize=(10, 5.2))
order = ["fold1_train", "fold1_valid", "fold1_test", "fold2_test", "fold3_test"]
width = 0.36
x = np.arange(len(order))
for offset, bucket, family_name, label in [(-width/2, "01", "blue", "bucket 01"), (width/2, "31", "orange", "bucket 31")]:
    part = period_top_bottom[period_top_bottom["bucket_label_2d"].eq(bucket)].set_index("period_label").loc[order]
    ax.bar(x + offset, part["ann_return"], width=width, label=label, color=COLOR_FAMILIES[family_name]["base"], edgecolor=COLOR_FAMILIES[family_name]["dark"], linewidth=1.0)
ax.axhline(0, color=TOKENS["ink"], linewidth=1.0)
ax.set_xticks(x, order, rotation=20, ha="right")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.set_ylabel("Annualized return")
ax.grid(axis="y", color=TOKENS["grid"])
ax.legend(loc="lower left", bbox_to_anchor=(0, 1.02), frameon=False, ncol=2, borderaxespad=0)
add_chart_header(
    fig,
    ax,
    "Top and bottom bucket returns differ across train, validation, and test windows",
    "Fold1 train/valid are diagnostic segments; only fold test windows should be described as rolling test performance.",
)
plt.show()


## 04 Appendix and reproducibility notes

Recommended smoke-test command after editing the notebook or core scripts:

```powershell
.\.venv\Scripts\python.exe -m pytest tests\test_build_lgbm_non_neutralized_training_data.py tests\test_train_lgbm_placeholder_model.py tests\test_evaluate_lgbm_training_metrics.py tests\test_evaluate_lgbm_rank_bucket_nav.py tests\test_build_chronological_rank_bucket_predictions.py tests\test_summarize_rank_bucket_periods.py
```

Notebook execution with `nbconvert` requires extra notebook dependencies that are not part of the current `pyproject.toml`. In the current project environment, this notebook is designed so its code cells can also be smoke-tested by extracting and executing them with plain Python.


In [ ]:
optional_artifacts = []
for label, path in {
    "alpha comparison summary": ALPHA_COMPARE_DIR / "alpha_signal_rank_bucket_comparison_summary.json",
    "alpha comparison requested table": ALPHA_COMPARE_DIR / "rank_bucket_period_requested_table.csv",
    "model training report html": MODEL_DIR / "training_report.html",
    "full 31-bucket chronological png": RB_CHRONO_DIR / "charts" / "rank_bucket_nav_fold_0_full_chronological.png",
}.items():
    optional_artifacts.append({"artifact": label, "path": str(path.relative_to(REPO_ROOT)), "exists": path.exists()})
show(pd.DataFrame(optional_artifacts), max_rows=20)

known_limits = pd.DataFrame(
    [
        {"item": "Costs", "note": "Rank-bucket NAV is gross_no_cost; no trading fee, slippage, or turnover cost is deducted."},
        {"item": "Full chronological NAV", "note": "Fold1 train/valid portions are diagnostic, not pure OOS."},
        {"item": "Monotone constraints", "note": "Only local partial monotonicity on alpha rank/z with all other features fixed."},
        {"item": "score_marginal_z", "note": "Current IC metric tables have zero valid date_count; do not use as headline score."},
        {"item": "Feature gain", "note": "LightGBM split/gain statistic, not causal feature contribution."},
    ]
)
show(known_limits, max_rows=20)
